# Vision Situation Analysis Review

Review notebook for the updated Vision/DL direction after the meeting.

This notebook checks event-window candidates, key-frame roles, object changes, evidence images, and grounded summary text. It does not validate fault ratio, legal liability, or final accident type.


## 1. Check project paths and generated outputs

Find the project root and count existing detection, agent-output, and visualization files.


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "storage/vision/outputs"
DETECTION_DIR = OUTPUT_DIR / "detections"
AGENT_OUTPUT_DIR = OUTPUT_DIR / "agent_outputs"
VIS_DIR = OUTPUT_DIR / "visualizations"

print("project_root:", PROJECT_ROOT)
print("detections:", len(list(DETECTION_DIR.glob("detections_*.json"))))
print("agent_outputs:", len(list(AGENT_OUTPUT_DIR.glob("agent_output_*.json"))))
print("visualizations:", len(list(VIS_DIR.glob("*_bbox.jpg"))))


## 1-A. Run full Vision pipeline

Run this cell when you want the notebook to regenerate all outputs from the current raw video. It executes the same `.py` files that were previously run in the RunPod terminal.


In [ ]:
# AUTO_RUN_FULL_VISION_PIPELINE
import subprocess
import sys

RUN_FULL_PIPELINE = True

commands = [
    [sys.executable, "scripts/check_raw_media.py"],
    [sys.executable, "ai/vision/pipeline.py"],
    [sys.executable, "ai/vision/models.py"],
    [sys.executable, "ai/vision/schemas.py"],
    [sys.executable, "ai/vision/visualize.py"],
    [sys.executable, "etl/build_clip_candidates.py", "--short-video-sec", "10"],
    [sys.executable, "etl/extract_video_clips.py", "--overwrite"],
    [sys.executable, "etl/extract_videomae_frames.py", "--overwrite"],
    [sys.executable, "ai/vision/videomae_infer.py"],
    [sys.executable, "ai/vision/merge_analysis.py"],
    [sys.executable, "ai/vision/build_supervisor_handoff.py"],
]

if RUN_FULL_PIPELINE:
    for command in commands:
        print("\n$", " ".join(command))
        completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True)
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr)
        completed.check_returncode()
else:
    print("RUN_FULL_PIPELINE is False. Set it to True to regenerate outputs.")


## 2. Load latest Vision Agent Output

Load the latest JSON generated by `ai/vision/schemas.py`. This is the structure passed to RAG/report/UI layers.


In [ ]:
def latest(pattern_dir, pattern):
    files = sorted(pattern_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files found: {pattern_dir / pattern}")
    return files[-1]

agent_output_path = latest(AGENT_OUTPUT_DIR, "agent_output_*.json")
agent_data = json.loads(agent_output_path.read_text(encoding="utf-8"))
agent = agent_data["agent_output"]
result = agent["structured_result"]

print("agent_output_path:", agent_output_path)
print("schema_version:", agent["metadata"]["schema_version"])
print("node_code:", agent["node_code"])
print("status:", agent["status"])
print("summary:", agent["summary"])


## 3. Review event-window candidates

These candidates are produced from bbox motion, object appearance/disappearance, and key-frame timestamp context.


In [ ]:
event_windows = result.get("event_window_candidates", [])
for item in event_windows:
    print(json.dumps(item, ensure_ascii=False, indent=2))


## 4. Review key-frame roles

Check whether each frame is treated as event_before, risk_increase, event_peak, event_after, or fallback.


In [ ]:
for frame in result.get("key_frames", []):
    print(
        frame.get("frame_id"),
        "role=" + str(frame.get("frame_role")),
        "time=" + str(frame.get("timestamp_sec")),
        "reason=" + str(frame.get("selection_reason")),
        "path=" + str(frame.get("frame_path")),
    )


## 5. Review object-change evidence

Check motion score, bbox center movement, area change, appeared classes, and disappeared classes between sampled frames.


In [ ]:
changes = result.get("object_change_observations", [])
print("change_count:", len(changes))
for item in changes[:10]:
    print(json.dumps(item, ensure_ascii=False, indent=2))


## 6. Review detected objects

Check object-level class, confidence, and bbox. There is no single generic confidence for the whole analysis.


In [ ]:
objects = result.get("detected_objects", [])
print("detected_objects:", len(objects))
for obj in objects[:20]:
    print(obj["source_ref"], obj["class_name"], obj["confidence"], obj["bbox"]["values"])


## 7. Display bbox visualization images

These images are useful for review and presentation. Run `ai/vision/visualize.py` first if none are shown.


In [ ]:
vis_images = sorted(VIS_DIR.glob("*_bbox.jpg"))
print("visualization_images:", len(vis_images))
for image_path in vis_images[:5]:
    print(image_path)
    display(Image(filename=str(image_path), width=640))


## 8. Review grounded summary and unavailable items

The field summary must only use observed evidence. Fault ratio, liable party, and legal responsibility stay unavailable here.


In [ ]:
print("[field_summary]")
print(result.get("field_summary"))

print("\n[unavailable_items]")
for item in result.get("unavailable_items", []):
    print("-", item["item"], ":", item["reason"])

print("\n[limitations]")
for item in result.get("limitations", []):
    print("-", item["type"], ":", item["message"])


## 9. Regenerate Agent Output when needed

Use this only after changing `ai/vision/schemas.py`. It keeps the same detection JSON and regenerates the normalized output.


In [ ]:
# Run only when needed.
# import sys
# !{sys.executable} {PROJECT_ROOT / 'ai/vision/schemas.py'}


## 10. Review extracted event clips

Load clip candidate JSON and extracted mp4 clips. These clips are the local input candidates for later VideoMAE comparison POC.


In [ ]:
from IPython.display import Video

CLIP_CANDIDATE_DIR = OUTPUT_DIR / "clip_candidates"
CLIP_DIR = PROJECT_ROOT / "storage/vision/processed/clips"

clip_candidate_path = latest(CLIP_CANDIDATE_DIR, "clip_candidates_*.json")
clip_candidate_data = json.loads(clip_candidate_path.read_text(encoding="utf-8"))

print("clip_candidate_path:", clip_candidate_path)
print("candidate_count:", clip_candidate_data.get("candidate_count"))
for item in clip_candidate_data.get("clip_candidates", []):
    print(json.dumps(item, ensure_ascii=False, indent=2))


## 11. Play extracted clips

If the video does not render in VS Code, open the printed mp4 path directly from the file explorer.


In [ ]:
import cv2
from pathlib import Path
from IPython.display import Image, display

clip_path = PROJECT_ROOT / "storage/vision/processed/clips/bb_3_190909_pedestrian_226_21450_clip_01.mp4"
preview_dir = PROJECT_ROOT / "storage/vision/processed/clips/preview_frames"
preview_dir.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(clip_path))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("frame_count:", frame_count)

indices = [0, frame_count // 4, frame_count // 2, frame_count * 3 // 4, frame_count - 1]

for i, idx in enumerate(indices, start=1):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if not ok:
        print("failed:", idx)
        continue

    out_path = preview_dir / f"clip_preview_{i:02d}_{idx:06d}.jpg"
    cv2.imwrite(str(out_path), frame)

    print(out_path)
    display(Image(filename=str(out_path), width=640))

cap.release()

In [ ]:
import subprocess
from pathlib import Path
import imageio_ffmpeg

ffmpeg_exe = imageio_ffmpeg.get_ffmpeg_exe()

clip_path = PROJECT_ROOT / "storage/vision/processed/clips/bb_3_190909_pedestrian_226_21450_clip_01.mp4"
h264_path = clip_path.with_name(clip_path.stem + "_h264.mp4")

command = [
    ffmpeg_exe,
    "-y",
    "-i", str(clip_path),
    "-vcodec", "libx264",
    "-pix_fmt", "yuv420p",
    "-acodec", "aac",
    str(h264_path),
]

result = subprocess.run(command, text=True, capture_output=True)

print("returncode:", result.returncode)
print("ffmpeg:", ffmpeg_exe)
print("h264_path:", h264_path)
print("h264_exists:", h264_path.exists())

if result.returncode != 0:
    print(result.stderr)

In [ ]:
from IPython.display import Video, display

display(Video(str(h264_path), embed=True, width=640))

## 12. Review VideoMAE input frames

Check the 16 uniformly sampled frames that will be used as VideoMAE comparison POC input.


In [ ]:
VIDEOMAE_INPUT_DIR = OUTPUT_DIR / "videomae_inputs"
videomae_manifest_path = latest(VIDEOMAE_INPUT_DIR, "videomae_clip_manifest_*.json")
videomae_data = json.loads(videomae_manifest_path.read_text(encoding="utf-8"))

print("videomae_manifest_path:", videomae_manifest_path)
print("clip_count:", videomae_data.get("clip_count"))
print("target_frame_count:", videomae_data.get("target_frame_count"))

for clip in videomae_data.get("clips", []):
    print("clip_id:", clip.get("clip_id"))
    print("clip_path:", clip.get("clip_path"))
    print("metadata:", clip.get("metadata"))


## 13. Display VideoMAE sampled frames

Show all sampled frames in order. These are the frames a later VideoMAE script will consume.


In [ ]:
for clip in videomae_data.get("clips", []):
    frames = clip.get("videomae_input", {}).get("frames", [])
    print("clip_id:", clip.get("clip_id"), "frames:", len(frames))
    for frame in frames:
        frame_path = PROJECT_ROOT / frame["frame_path"]
        print(frame["frame_order"], frame["timestamp_sec"], frame_path)
        if frame_path.exists():
            display(Image(filename=str(frame_path), width=360))


## 14. Review final merged analysis

Confirm the final output that combines YOLO/bbox evidence and VideoMAE clip-level inference.


In [ ]:
# AUTO_REVIEW_FINAL_ANALYSIS
FINAL_DIR = OUTPUT_DIR / "final_analysis"
VIDEOMAE_RESULT_DIR = OUTPUT_DIR / "videomae_results"

try:
    final_analysis_path = latest(FINAL_DIR, "final_analysis_*.json")
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "final_analysis_*.json이 아직 없습니다. 위의 'Run full Vision pipeline' 셀을 먼저 끝까지 실행하세요. "
        "특히 ai/vision/videomae_infer.py 와 ai/vision/merge_analysis.py 단계가 성공해야 합니다."
    ) from exc
final_data = json.loads(final_analysis_path.read_text(encoding="utf-8"))

agent = final_data["vision_agent_output"]["agent_output"]
structured = agent["structured_result"]
video = final_data["video_understanding"]

print("final_analysis_path:", final_analysis_path)
print("schema_version:", final_data.get("schema_version"))
print("status:", final_data.get("status"))
print("summary:", agent.get("summary"))
print("event_windows:", len(structured.get("event_window_candidates", [])))
print("key_frames:", len(structured.get("key_frames", [])))
print("detected_objects:", len(structured.get("detected_objects", [])))

for clip in video.get("clips", []):
    top = clip.get("top_prediction") or {}
    print("videomae:", clip.get("clip_id"), top.get("label"), top.get("score"))

print("\nlimitations")
for item in final_data.get("limitations", []):
    print("-", item)


## 15. Review Supervisor handoff

Run this after the full pipeline. It verifies the lightweight schema that Supervisor can pass to legal, precedent, RAG, and report agents.


In [ ]:
# AUTO_REVIEW_SUPERVISOR_HANDOFF
HANDOFF_DIR = OUTPUT_DIR / "supervisor_handoff"
handoff_path = latest(HANDOFF_DIR, "vision_supervisor_handoff_*.json")
handoff_data = json.loads(handoff_path.read_text(encoding="utf-8"))
handoff = handoff_data["vision_supervisor_handoff"]

print("handoff_path:", handoff_path)
print("schema_version:", handoff.get("schema_version"))
print("status:", handoff.get("status"))
print("source:", handoff.get("source"))

print("\nmedia_summary")
for key, value in handoff.get("media_summary", {}).items():
    print(f"- {key}: {value}")

print("\nevent_candidates:", len(handoff.get("event_candidates", [])))
for event in handoff.get("event_candidates", []):
    print("-", event)

visual = handoff.get("visual_evidence", {})
print("\nvisual_evidence")
print("- key_frames:", len(visual.get("key_frames", [])))
print("- evidence_candidates:", len(visual.get("evidence_candidates", [])))
print("- detected_object_summary:", visual.get("detected_object_summary"))

print("\nvideo_understanding_hint")
for key, value in handoff.get("video_understanding_hint", {}).items():
    print(f"- {key}: {value}")

print("\nnot_determined_by_vision")
for item in handoff.get("not_determined_by_vision", []):
    print("-", item)

print("\nrouting_recommendation")
for key, value in handoff.get("routing_recommendation", {}).items():
    print(f"- {key}: {value}")
